In [1]:
import numpy as np
import pandas as pd
from src import Replicator

In [2]:
methods = ['cs', '4p', 'fw']
estimators = ['x', 'mu', 'var']

In [21]:
def repeatCompare(fn, R, n, param, tau, cstr=np.array([10]*2), sclr=1):
    rslt = {}
    threads = {key: {} for key in methods}
    for key in methods:
        for i in range(12):
            name = '{}#{}#'.format(key,i+1)
            threads[key][name] = Replicator(fn, rslt, key, R, n, param[key], tau, cstr, sclr, name)
    
    for key in methods:
        for name in threads[key]:
            threads[key][name].start()

    for key in methods:
        for name in threads[key]:
            threads[key][name].join()

    return rslt

# Objective Functions

In [4]:
L = np.array([
    [2, 0],
    [-1, 2]
])
A = L @ L.T
x = np.array([1, -2])
b = A @ x
c = 20
y = x @ A @ x / 2 - b @ x + c

def mu_fn(x, c=0): return x @ A @ x / 2 - b @ x + c

def sigmoid(y): return 1 / (1 + np.exp(-y))

## Continuous

In [5]:
def fn_c1(x, tau=1):
    mu = mu_fn(x, c=20)
    sclr = 1 + np.linalg.norm(x) * np.cos(np.pi * np.linalg.norm(x))
    noise = np.random.normal(size=tau)
    return mu + sclr * np.mean(noise)

def fn_c2(x, tau=1):
    mu = 20 * sigmoid(mu_fn(x)/800 - 2)
    sclr = 1 + np.linalg.norm(x) * np.cos(np.pi * np.linalg.norm(x))
    noise = np.random.normal(size=tau)
    return mu + sclr * np.mean(noise)

## Discrete/Binary

In [6]:
def fn_b1(x, tau=1):
    p = min(.99, max(.01, mu_fn(x)/2000 + .3))
    return np.random.binomial(tau, p) / tau

def fn_b2(x, tau=1):
    p = min(.99, max(.01, 2 * sigmoid(mu_fn(x)/600 - 2)))
    return np.random.binomial(tau, p) / tau

# Preliminaries
## Parameters

In [7]:
R = 25; n = int(1e5)
param = {'cs': .05, '4p': np.sqrt(3/2), "fw":0}
tau = 3, 2
cstr = np.array([10]*2)
sclr = 100

In [8]:
fn = {
    'c1': fn_c1,
    'c2': fn_c2,
    'b1': fn_b1,
    'b2': fn_b2
}
# methods = ['cs', '4p']
# estimators = ['x', 'mu', 'var']

# Main

In [15]:
for ftp in fn:
# for ftp in ['c1', 'b2']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in ['fw']:
    # for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/FW/Exp_{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#5# finishes replication 1 and takes 497968.71 ms
	final estimation x=(1.0882283202817138, -1.998510174206439), mu=4.16 and var=108.19
Thread cs#7# finishes replication 1 and takes 497999.58 ms
	final estimation x=(1.1544243346605536, -1.7973740009180792), mu=3.98 and var=119.72
Thread cs#2# finishes replication 1 and takes 498160.87 ms
	final estimation x=(1.0961216375026583, -1.9319262254243939), mu=4.13 and var=129.16
Thread cs#3# finishes replication 1 and takes 498220.60 ms
	final estimation x=(0.8227051626240627, -2.0047718609452208), mu=4.11 and var=103.50
Thread cs#6# finishes replication 1 and takes 498280.34 ms
	final estimation x=(1.0465368585774162, -1.997757653757072), mu=4.11 and var=108.56
Thread cs#1# finishes replication 1 and takes 498318.17 ms
	final estimation x=(1.1777085322375933, -2.0693019571636664), mu=4.13 and var=124.63
Thread cs#4# finishes replication 1 and takes 498613.87 ms
	final estimation x=(1.1063253374753292, -1.9697671151322593), mu=4.02 an

In [16]:
# for ftp in fn:
for ftp in ['c2', 'b1']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/d2Exp#A#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#1# finishes replication 1 and takes 520151.92 ms
	final estimation x=(3.2081823803214378, 0.952965519966546), mu=2.44 and var=0.16
Thread cs#7# finishes replication 1 and takes 520198.71 ms
	final estimation x=(0.39462697739525326, -1.5285261216669237), mu=2.36 and var=0.29
Thread cs#6# finishes replication 1 and takes 520296.28 ms
	final estimation x=(3.664791327566713, -2.8210866893138884), mu=2.49 and var=0.43
Thread cs#2# finishes replication 1 and takes 520433.68 ms
	final estimation x=(1.1436491725537845, -0.8295970291427851), mu=2.37 and var=0.25
Thread cs#5# finishes replication 1 and takes 520494.41 ms
	final estimation x=(2.231330737213235, -0.8301833959003591), mu=2.61 and var=0.28
Thread cs#3# finishes replication 1 and takes 520669.64 ms
	final estimation x=(2.1310499570417263, -2.569921212403225), mu=2.39 and var=0.17
Thread cs#4# finishes replication 1 and takes 520738.33 ms
	final estimation x=(0.4916259854108025, -2.5786887995448438), mu=2.41 and var=0.13
Thr

In [17]:
R = 25; n = int(1e4)
param = {'cs': .05, '4p': np.sqrt(3/2)}
tau = 3, 2
cstr = np.array([10]*2)
sclr = 1000

for ftp in ['b1', 'b2']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/L/d2Exp#L#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#7# finishes replication 1 and takes 30745.46 ms
	final estimation x=(0.6732860779683083, 1.428856367457334), mu=0.27 and var=0.02
Thread cs#6# finishes replication 1 and takes 30784.29 ms
	final estimation x=(-8.164144100177428, -5.709160602627013), mu=0.39 and var=0.02
Thread cs#5# finishes replication 1 and takes 30800.22 ms
	final estimation x=(0.05614718475814047, -0.8307964110725986), mu=0.29 and var=0.02
Thread cs#1# finishes replication 1 and takes 30861.94 ms
	final estimation x=(-3.64597648338164, -0.29703627665296256), mu=0.32 and var=0.02
Thread cs#4# finishes replication 1 and takes 30937.61 ms
	final estimation x=(0.9193659117267047, -1.7322409066830406), mu=0.29 and var=0.02
Thread cs#2# finishes replication 1 and takes 31012.28 ms
	final estimation x=(0.6733577161876348, -0.30971710539327474), mu=0.31 and var=0.02
Thread cs#3# finishes replication 1 and takes 31129.76 ms
	final estimation x=(8.496002334804551, 3.7166110327094892), mu=0.36 and var=0.02
Thread 4p

In [20]:
R = 25; n = int(1e5)
param = {'cs': .05, '4p': np.sqrt(3/2)}
tau = 3, 2
cstr = np.array([10]*2)
sclr = 1000

for ftp in ['b1', 'b2']:
    cache = repeatCompare(fn[ftp], R, n, param, tau, cstr, sclr)
    for key in methods:
        for each in estimators:
            cache[key][each].to_csv("./output/L/d2Exp#C7#{}_{}_{}.csv".format(ftp, key, each), index=False)

Thread cs#3# finishes replication 1 and takes 249808.73 ms
	final estimation x=(1.8972219648626047, -2.5218154765491763), mu=0.31 and var=0.02
Thread cs#4# finishes replication 1 and takes 249883.39 ms
	final estimation x=(8.508270383555347, -0.4822736047352801), mu=0.37 and var=0.02
Thread cs#5# finishes replication 1 and takes 250016.81 ms
	final estimation x=(4.313226178948754, -0.7149011662594686), mu=0.31 and var=0.02
Thread cs#2# finishes replication 1 and takes 250223.89 ms
	final estimation x=(4.369130762209827, 0.2944968105461002), mu=0.32 and var=0.02
Thread cs#1# finishes replication 1 and takes 250668.93 ms
	final estimation x=(3.4966341842656363, 0.26449602302814085), mu=0.27 and var=0.02
Thread 4p#1# finishes replication 1 and takes 389711.96 ms
	final estimation x=(-8.716308440455075, -3.3675507694415265), mu=0.36 and var=0.57
Thread 4p#2# finishes replication 1 and takes 389895.15 ms
	final estimation x=(-9.076980351069043, -0.3715902598089495), mu=0.37 and var=0.58
Thr